# Lab 1 · Pregúntele a sus números
### ISM Snow Day · Gerentes · 20 minutos · sin SQL

**El problema.** Cada cierre de mes, gerencia hace las mismas preguntas: ¿cuánto vendimos?, ¿qué margen dejamos?, ¿cómo quedó el EBITDA?, ¿cuánto se nos fue en descuentos y bonificaciones?, ¿cuánto efectivo entró y salió? Hoy cada respuesta requiere pedir un reporte, esperar y cruzar archivos.

**Lo que va a lograr.** En 20 minutos va a tener un **asistente de IA en Snowflake CoWork** que responde esas preguntas en español, con cifras que cuadran, sobre un modelo financiero de ISM: ventas, margen, estado de resultados, inventario y flujo de efectivo.

| Paso | Qué hace | Tiempo |
|---|---|---|
| 1 | Ejecutar las celdas de preparación (datos, modelo semántico y agente) | 5 min |
| 2 | Abrir CoWork y hacerle al asistente las 9 preguntas de gerencia | 12 min |
| 3 | Comparar con las respuestas esperadas y pedir un resumen ejecutivo | 3 min |

> **Datos 100% sintéticos.** Las tablas imitan la estructura de ISM (sociedades SAP, ventas, bonificaciones, inventario MB52), pero las marcas, clientes y cifras son ficticios.

## Paso 1 · Preparación (5 min)

Ejecute las celdas en orden con el botón ▶ (o **Shift + Enter**). No necesita escribir nada.

**1.1 · Sesión.** Crea su warehouse, su base de datos `ISM_SNOWDAY` y su schema `LAB1`.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Usted es dueño de esta cuenta (ACCOUNTADMIN), así que el lab crea todo lo que necesita.
session.sql("USE ROLE ACCOUNTADMIN").collect()

db = "ISM_SNOWDAY"
schema = "LAB1"
fq = f"{db}.{schema}"
wh = "ISM_SNOWDAY_WH"

for stmt in [
    # Warehouse pequeño para las consultas, las funciones de IA y el agente
    f"CREATE WAREHOUSE IF NOT EXISTS {wh} WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE",
    f"USE WAREHOUSE {wh}",
    # Permite usar modelos de IA de otra región si alguno no está disponible en la región de esta cuenta
    "ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION'",
    f"CREATE DATABASE IF NOT EXISTS {db}",
    f"CREATE SCHEMA IF NOT EXISTS {fq}",
    f"USE SCHEMA {fq}",
]:
    session.sql(stmt).collect()

print(f"Rol        : {session.get_current_role()}")
print(f"Su schema  : {fq}")
print(f"Warehouse  : {wh}")

**1.2 · Datos.** Crea 7 tablas con 18 meses de historia para 5 sociedades: ISM Dominicana, Guatemala, Haití, Perú y Chile.

| Tabla | Qué contiene |
|---|---|
| `DIM_SOCIEDAD`, `DIM_PRODUCTO`, `DIM_CLIENTE` | Sociedades, productos (7 marcas, 6 categorías) y clientes por canal |
| `FACT_DETALLE_VENTAS` | Venta bruta, descuentos, bonificaciones, rechazos, venta neta y costo por cliente y SKU |
| `ESTADO_RESULTADOS_MENSUAL` | Estado de resultados por sociedad y mes |
| `INVENTARIOS_MB52` | Foto de inventario a fin de mes por material y tipo de material |
| `FLUJO_EFECTIVO` | Entradas y salidas de efectivo por concepto |

In [ ]:
# Genera los datos sintéticos de ISM: mismas tablas y columnas que usa ISM (estilo SAP), con marcas y clientes ficticios.
# No necesita leer esta celda: solo ejecútela (tarda alrededor de 1 minuto).

SOCIEDADES = [
    # id, bukrs, nombre, id_pais, pais, moneda, fx (moneda por USD), peso
    (1000, "DO01", "ISM Dominicana", "DO", "República Dominicana", "DOP", 58.9, 0.42),
    (2000, "GT01", "ISM Guatemala", "GT", "Guatemala", "GTQ", 7.75, 0.22),
    (3000, "HT01", "ISM Haití", "HT", "Haití", "HTG", 132.0, 0.10),
    (4000, "PE01", "ISM Perú", "PE", "Perú", "PEN", 3.74, 0.18),
    (5000, "CL01", "ISM Chile", "CL", "Chile", "CLP", 935.0, 0.08),
]

PRODUCTOS = [
    # sku, producto, marca, categoria, formato, litros_por_caja, precio_caja_usd, costo_caja_usd
    ("PT1001", "Kola Isla Lata 355ml", "Kola Isla", "Gaseosas", "355ml", 8.52, 9.60, 5.30),
    ("PT1002", "Kola Isla 600ml", "Kola Isla", "Gaseosas", "600ml", 14.40, 10.80, 5.90),
    ("PT1003", "Kola Isla 1.5L", "Kola Isla", "Gaseosas", "1.5L", 18.00, 11.40, 6.80),
    ("PT1004", "Kola Isla 2L", "Kola Isla", "Gaseosas", "2L", 16.00, 9.90, 6.10),
    ("PT1005", "Kola Isla 3L", "Kola Isla", "Gaseosas", "3L", 18.00, 10.20, 6.60),
    ("PT1101", "Fresquita Naranja 600ml", "Fresquita", "Gaseosas", "600ml", 14.40, 10.20, 5.70),
    ("PT1102", "Fresquita Naranja 2L", "Fresquita", "Gaseosas", "2L", 16.00, 9.30, 5.90),
    ("PT1103", "Fresquita Uva 600ml", "Fresquita", "Gaseosas", "600ml", 14.40, 10.20, 5.70),
    ("PT1104", "Fresquita Uva 2L", "Fresquita", "Gaseosas", "2L", 16.00, 9.30, 5.90),
    ("PT1105", "Fresquita Limón 600ml", "Fresquita", "Gaseosas", "600ml", 14.40, 10.20, 5.70),
    ("PT1106", "Fresquita Limón 2L", "Fresquita", "Gaseosas", "2L", 16.00, 9.30, 5.90),
    ("PT2001", "Agua Montaña 600ml", "Agua Montaña", "Agua", "600ml", 14.40, 6.00, 2.60),
    ("PT2002", "Agua Montaña 1L", "Agua Montaña", "Agua", "1L", 12.00, 5.40, 2.40),
    ("PT2003", "Agua Montaña 5L", "Agua Montaña", "Agua", "5L", 20.00, 4.80, 2.30),
    ("PT2004", "Agua Montaña con Gas 600ml", "Agua Montaña", "Agua", "600ml", 14.40, 7.20, 3.20),
    ("PT3001", "Rayo Energy Lata 250ml", "Rayo", "Energizantes", "250ml", 6.00, 16.80, 8.10),
    ("PT3002", "Rayo Energy Lata 473ml", "Rayo", "Energizantes", "473ml", 11.35, 25.20, 11.90),
    ("PT3003", "Rayo Zero Lata 473ml", "Rayo", "Energizantes", "473ml", 11.35, 25.20, 12.20),
    ("PT4001", "Frutalia Naranja 1L", "Frutalia", "Jugos y Néctares", "1L", 12.00, 13.20, 7.40),
    ("PT4002", "Frutalia Durazno 250ml", "Frutalia", "Jugos y Néctares", "250ml", 6.00, 9.60, 5.40),
    ("PT4003", "Frutalia Mango 1L", "Frutalia", "Jugos y Néctares", "1L", 12.00, 13.20, 7.50),
    ("PT5001", "Hidra+ Mora 500ml", "Hidra+", "Isotónicos", "500ml", 6.00, 9.00, 4.40),
    ("PT5002", "Hidra+ Limón 500ml", "Hidra+", "Isotónicos", "500ml", 6.00, 9.00, 4.40),
    ("PT5003", "Hidra+ Tropical 1L", "Hidra+", "Isotónicos", "1L", 12.00, 11.40, 5.80),
    ("PT6001", "Té Brisa Limón 500ml", "Té Brisa", "Té", "500ml", 6.00, 8.40, 4.30),
    ("PT6002", "Té Brisa Durazno 500ml", "Té Brisa", "Té", "500ml", 6.00, 8.40, 4.30),
]

# Non-finished-goods materials for the MB52 inventory snapshot: material, descripcion, tipo, unidad, costo_usd, cantidad_base
MATERIALES = [
    ("MP0001", "Azúcar refinada", "Materia prima", "KG", 0.62, 180000),
    ("MP0002", "Concentrado sabor cola", "Materia prima", "L", 21.50, 2600),
    ("MP0003", "Concentrado sabor frutas", "Materia prima", "L", 18.40, 2100),
    ("MP0004", "CO2 grado alimenticio", "Materia prima", "KG", 0.48, 52000),
    ("MP0005", "Pulpa de fruta", "Materia prima", "KG", 1.35, 24000),
    ("MP0006", "Cafeína y taurina (premezcla)", "Materia prima", "KG", 38.00, 420),
    ("EN0001", "Preforma PET 23g", "Envases y empaques", "UN", 0.031, 2600000),
    ("EN0002", "Preforma PET 48g", "Envases y empaques", "UN", 0.058, 900000),
    ("EN0003", "Tapa plástica 28mm", "Envases y empaques", "UN", 0.006, 3800000),
    ("EN0004", "Lata de aluminio 473ml", "Envases y empaques", "UN", 0.115, 640000),
    ("EN0005", "Etiqueta termoencogible", "Envases y empaques", "UN", 0.009, 3100000),
    ("EN0006", "Caja de cartón corrugado", "Envases y empaques", "UN", 0.42, 160000),
    ("RP0001", "Rodamientos línea de llenado", "Repuestos", "UN", 46.00, 310),
    ("RP0002", "Válvula llenadora", "Repuestos", "UN", 128.00, 140),
    ("RP0003", "Banda transportadora", "Repuestos", "UN", 410.00, 36),
    ("PR0001", "Hielera con logo", "Material promocional", "UN", 22.00, 1800),
    ("PR0002", "Exhibidor de punto de venta", "Material promocional", "UN", 64.00, 650),
]


def _values(rows):
    out = []
    for r in rows:
        vals = []
        for v in r:
            if isinstance(v, str):
                vals.append("'" + v.replace("'", "''") + "'")
            else:
                vals.append(str(v))
        out.append("(" + ", ".join(vals) + ")")
    return ",\n".join(out)


def load_statements(fq: str, n_clientes: int = 700):
    """SQL statements that (re)build the shared ISM model in schema fq."""
    soc = _values(SOCIEDADES)
    prod = _values(PRODUCTOS)
    mat = _values(MATERIALES)
    return [
        # ---------- Dimensions ----------
        f"""CREATE OR REPLACE TABLE {fq}.DIM_SOCIEDAD (
              ID_SOCIEDAD NUMBER, BUKRS_SOCIEDAD STRING, SOCIEDAD STRING, ID_PAIS STRING, PAIS STRING,
              MONEDA STRING, TIPO_CAMBIO_USD NUMBER(12,4), PESO_NEGOCIO NUMBER(5,2))
            COMMENT = 'Sociedades (company codes SAP) de ISM. Datos sintéticos.'
            AS SELECT * FROM VALUES {soc}""",
        f"""CREATE OR REPLACE TABLE {fq}.DIM_PRODUCTO (
              ID_SKU_PRODUCTO STRING, PRODUCTO STRING, MARCA STRING, CATEGORIA STRING, FORMATO STRING,
              LITROS_POR_CAJA NUMBER(8,2), PRECIO_LISTA_CAJA_USD NUMBER(10,2), COSTO_ESTANDAR_CAJA_USD NUMBER(10,2))
            COMMENT = 'Maestro de producto terminado (SKU). Marcas ficticias.'
            AS SELECT * FROM VALUES {prod}""",
        f"""CREATE OR REPLACE TABLE {fq}.DIM_CLIENTE COMMENT = 'Maestro de clientes por sociedad y canal. Datos sintéticos.' AS
            WITH g AS (SELECT SEQ4() + 1 AS n FROM TABLE(GENERATOR(ROWCOUNT => {n_clientes}))),
            b AS (
              SELECT n,
                     MOD(ABS(HASH(n, 'soc')), 100) AS hs,
                     MOD(ABS(HASH(n, 'canal')), 100) AS hc,
                     MOD(ABS(HASH(n, 'nom')), 12) AS hn,
                     MOD(ABS(HASH(n, 'ciu')), 4) AS hci,
                     0.4 + MOD(ABS(HASH(n, 'tam')), 160) / 100 AS tam
              FROM g)
            SELECT 'C' || LPAD(n, 6, '0') AS ID_CUENTA,
                   CASE WHEN hs < 42 THEN 1000 WHEN hs < 64 THEN 2000 WHEN hs < 74 THEN 3000 WHEN hs < 92 THEN 4000 ELSE 5000 END AS ID_SOCIEDAD,
                   CASE WHEN hc < 62 THEN 'Tradicional' WHEN hc < 74 THEN 'Moderno' WHEN hc < 88 THEN 'Mayorista' ELSE 'On Premise' END AS CANAL_CORPORATIVO,
                   CASE WHEN hc < 62 THEN IFF(ID_SOCIEDAD = 1000, 'Colmado ', 'Tienda ') || ARRAY_CONSTRUCT('La Esperanza','El Progreso','Don Pedro','La Bendición','Los Hermanos','El Buen Precio','La Fe','San José','Doña María','El Sol','La Económica','Mi Barrio')[hn]::STRING
                        WHEN hc < 74 THEN ARRAY_CONSTRUCT('Supermercados Del Mar','Hiper Ahorro','Mercado Familiar','Super Express','Cadena Uno','Supermercado La Plaza','Mega Compras','Super Fresco','Plaza Market','Mercado Central','Super Norte','Hiper Sur')[hn]::STRING
                        WHEN hc < 88 THEN 'Distribuidora ' || ARRAY_CONSTRUCT('Del Norte','Caribe','Andina','Central','El Puerto','Del Valle Alto','La Unión','San Miguel Norte','El Cruce','Oriental','Los Pinos','Mercantil')[hn]::STRING
                        ELSE ARRAY_CONSTRUCT('Restaurante El Fogón','Bar La Terraza','Hotel Costa Azul','Cafetería Central','Pica Pollo Don Juan','Bar El Muelle','Hotel Plaza','Restaurante Sabores','Food Court Ágora','Cafetería La Esquina','Bar Los Amigos','Hotel del Parque')[hn]::STRING END
                   || ' ' || LPAD(n, 3, '0') AS NOMBRE_CLIENTE,
                   CASE ID_SOCIEDAD
                     WHEN 1000 THEN ARRAY_CONSTRUCT('Santo Domingo Oeste','Santo Domingo Este','Santiago','San Cristóbal')[hci]
                     WHEN 2000 THEN ARRAY_CONSTRUCT('Ciudad de Guatemala','Mixco','Quetzaltenango','Escuintla')[hci]
                     WHEN 3000 THEN ARRAY_CONSTRUCT('Puerto Príncipe','Cap-Haïtien','Pétion-Ville','Les Cayes')[hci]
                     WHEN 4000 THEN ARRAY_CONSTRUCT('Lima','Arequipa','Trujillo','Piura')[hci]
                     ELSE ARRAY_CONSTRUCT('Santiago de Chile','Valparaíso','Concepción','Antofagasta')[hci] END::STRING AS CIUDAD,
                   ROUND(tam * CASE WHEN hc < 62 THEN 1 WHEN hc < 74 THEN 9 WHEN hc < 88 THEN 16 ELSE 1.6 END, 2) AS TAMANO_RELATIVO
            FROM b
            UNION ALL
            SELECT 'C900001', 1000, 'Moderno', 'Hiper Ahorro Santo Domingo (cuenta clave)', 'Santo Domingo Oeste', 30""",
        # ---------- Sales fact (monthly, by client x SKU) ----------
        f"""CREATE OR REPLACE TABLE {fq}.FACT_DETALLE_VENTAS COMMENT = 'Detalle de ventas mensual por cliente y SKU (USD y moneda local). Datos sintéticos.' AS
            WITH meses AS (
              SELECT DATEADD(month, -(SEQ4()), DATE_TRUNC('month', CURRENT_DATE())) AS PERIODO, SEQ4() AS meses_atras
              FROM TABLE(GENERATOR(ROWCOUNT => 18))),
            surtido AS (
              SELECT c.*, p.*
              FROM {fq}.DIM_CLIENTE c CROSS JOIN {fq}.DIM_PRODUCTO p
              WHERE MOD(ABS(HASH(c.ID_CUENTA, p.ID_SKU_PRODUCTO)), 100) <
                    CASE c.CANAL_CORPORATIVO WHEN 'Moderno' THEN 85 WHEN 'Mayorista' THEN 70 WHEN 'On Premise' THEN 30 ELSE 32 END),
            base AS (
              SELECT s.ID_CUENTA, s.ID_SOCIEDAD, s.CANAL_CORPORATIVO, s.ID_SKU_PRODUCTO, s.CATEGORIA,
                     s.PRECIO_LISTA_CAJA_USD, s.COSTO_ESTANDAR_CAJA_USD, s.LITROS_POR_CAJA, m.PERIODO, m.meses_atras,
                     GREATEST(1, ROUND(
                       CASE s.CANAL_CORPORATIVO WHEN 'Moderno' THEN 56 WHEN 'Mayorista' THEN 72 WHEN 'On Premise' THEN 40 ELSE 48 END
                       * s.TAMANO_RELATIVO
                       * (1 + 0.12 * SIN((MONTH(m.PERIODO) - 3) * 3.14159 / 6))
                       * (1 - 0.006 * m.meses_atras)
                       * (0.75 + MOD(ABS(HASH(s.ID_CUENTA, s.ID_SKU_PRODUCTO, m.PERIODO)), 50) / 100)
                       * IFF(m.meses_atras = 0 AND s.ID_SKU_PRODUCTO = 'PT3002', 0.55, 1)
                       * IFF(m.meses_atras = 0 AND s.ID_SKU_PRODUCTO = 'PT4001', 0.72, 1)
                       * IFF(m.meses_atras = 0 AND s.ID_CUENTA = 'C900001', 0.42, 1)
                     )) AS CANTIDAD_CAJAS
              FROM surtido s CROSS JOIN meses m),
            calc AS (
              SELECT b.*,
                     ROUND(CANTIDAD_CAJAS * PRECIO_LISTA_CAJA_USD * (1 + 0.002 * (17 - meses_atras)), 2) AS USD_VALOR_BRUTO,
                     (CASE CANAL_CORPORATIVO WHEN 'Moderno' THEN 0.09 WHEN 'Mayorista' THEN 0.11 WHEN 'On Premise' THEN 0.03 ELSE 0.04 END
                       + MOD(ABS(HASH(ID_CUENTA, PERIODO, 'd')), 30) / 1000) AS pct_desc,
                     (CASE CANAL_CORPORATIVO WHEN 'Moderno' THEN 0.025 WHEN 'Mayorista' THEN 0.04 WHEN 'On Premise' THEN 0.05 ELSE 0.06 END
                       + IFF(meses_atras = 0 AND ID_SOCIEDAD = 1000, 0.025, 0)
                       + MOD(ABS(HASH(ID_CUENTA, PERIODO, 'b')), 20) / 1000) AS pct_bonif,
                     (0.008 + MOD(ABS(HASH(ID_CUENTA, ID_SKU_PRODUCTO, PERIODO, 'r')), 25) / 1000) AS pct_rech,
                     (1 + IFF(meses_atras <= 2, 0.04, 0) + (MOD(ABS(HASH(ID_SKU_PRODUCTO, PERIODO, 'c')), 40) - 20) / 1000) AS f_costo
              FROM base b)
            SELECT 'V' || LPAD(ROW_NUMBER() OVER (ORDER BY PERIODO, ID_CUENTA, ID_SKU_PRODUCTO), 8, '0') AS ID_VENTA,
                   DATEADD(day, MOD(ABS(HASH(ID_CUENTA, ID_SKU_PRODUCTO, PERIODO, 'f')), IFF(PERIODO = DATE_TRUNC('month', CURRENT_DATE()), DAY(CURRENT_DATE()), 28)), PERIODO) AS FECHA,
                   PERIODO, c.ID_SOCIEDAD, ID_CUENTA, ID_SKU_PRODUCTO, CANAL_CORPORATIVO, CATEGORIA AS CATEGORIA_VENTA,
                   CANTIDAD_CAJAS, ROUND(CANTIDAD_CAJAS * LITROS_POR_CAJA, 2) AS LITROS,
                   USD_VALOR_BRUTO,
                   ROUND(USD_VALOR_BRUTO * pct_desc, 2) AS USD_DESCUENTOS,
                   ROUND(USD_VALOR_BRUTO * pct_bonif, 2) AS USD_BONIFICACIONES,
                   ROUND(USD_VALOR_BRUTO * pct_rech, 2) AS USD_VALOR_RECHAZO,
                   ROUND(USD_VALOR_BRUTO * (1 - pct_desc - pct_bonif - pct_rech), 2) AS USD_VALOR_NETO,
                   ROUND(CANTIDAD_CAJAS * (1 - pct_rech) * COSTO_ESTANDAR_CAJA_USD * f_costo, 2) AS USD_COSTO_VENTA,
                   s.MONEDA,
                   ROUND(USD_VALOR_BRUTO * (1 - pct_desc - pct_bonif - pct_rech) * s.TIPO_CAMBIO_USD, 2) AS VALOR_NETO_ML
            FROM calc c JOIN {fq}.DIM_SOCIEDAD s ON s.ID_SOCIEDAD = c.ID_SOCIEDAD""",
        # ---------- Monthly P&L by company (estado de resultados) ----------
        f"""CREATE OR REPLACE TABLE {fq}.ESTADO_RESULTADOS_MENSUAL COMMENT = 'Estado de resultados mensual por sociedad (USD). Montos con signo: ingresos positivos, costos y gastos negativos.' AS
            WITH v AS (
              SELECT PERIODO, ID_SOCIEDAD, SUM(USD_VALOR_NETO) AS neto, SUM(USD_COSTO_VENTA) AS costo
              FROM {fq}.FACT_DETALLE_VENTAS GROUP BY 1, 2),
            l AS (
              SELECT * FROM VALUES
                (1, 'Ventas netas', 'Ingreso', TRUE, 0.0),
                (2, 'Costo de ventas', 'Costo', TRUE, 0.0),
                (3, 'Gastos de venta y mercadeo', 'Gasto operativo', TRUE, 0.095),
                (4, 'Gastos de distribución', 'Gasto operativo', TRUE, 0.078),
                (5, 'Gastos administrativos', 'Gasto operativo', TRUE, 0.052),
                (6, 'Depreciación y amortización', 'Depreciación', FALSE, 0.031),
                (7, 'Gastos financieros', 'Financiero', FALSE, 0.012)
                AS l(ORDEN, LINEA_RESULTADO, TIPO_LINEA, INCLUIDO_EN_EBITDA, pct)),
            pre AS (
              SELECT v.PERIODO, v.ID_SOCIEDAD, l.ORDEN, l.LINEA_RESULTADO, l.TIPO_LINEA, l.INCLUIDO_EN_EBITDA,
                     ROUND(CASE l.ORDEN WHEN 1 THEN v.neto WHEN 2 THEN -v.costo
                           ELSE -v.neto * l.pct * (0.92 + MOD(ABS(HASH(v.PERIODO, v.ID_SOCIEDAD, l.ORDEN)), 16) / 100)
                                 * IFF(l.ORDEN = 4 AND v.ID_SOCIEDAD = 3000, 1.35, 1) END, 2) AS USD_MONTO
              FROM v CROSS JOIN l)
            SELECT * FROM pre
            UNION ALL
            SELECT PERIODO, ID_SOCIEDAD, 8, 'Impuesto sobre la renta', 'Impuesto', FALSE,
                   ROUND(-GREATEST(SUM(USD_MONTO), 0) * 0.27, 2)
            FROM pre GROUP BY PERIODO, ID_SOCIEDAD""",
        # ---------- MB52-style inventory snapshot (month-end) ----------
        f"""CREATE OR REPLACE TABLE {fq}.INVENTARIOS_MB52 COMMENT = 'Foto de inventario a fin de mes por sociedad, centro y material (estilo SAP MB52). Valores en USD.' AS
            WITH meses AS (
              SELECT DATEADD(month, -(SEQ4()), DATE_TRUNC('month', CURRENT_DATE())) AS FECHA_PERIODO, SEQ4() AS meses_atras
              FROM TABLE(GENERATOR(ROWCOUNT => 18))),
            mat AS (
              SELECT ID_SKU_PRODUCTO AS MATERIAL, PRODUCTO AS DESCRIPCION, 'Producto terminado' AS TIPO_MATERIAL, 'CJ' AS UBM,
                     COSTO_ESTANDAR_CAJA_USD AS COSTO_USD, 9000 AS BASE
              FROM {fq}.DIM_PRODUCTO
              UNION ALL
              SELECT * FROM VALUES {mat} AS m(MATERIAL, DESCRIPCION, TIPO_MATERIAL, UBM, COSTO_USD, BASE)),
            x AS (
              SELECT m.FECHA_PERIODO, m.meses_atras, s.ID_SOCIEDAD, s.SOCIEDAD, s.ID_PAIS, s.PESO_NEGOCIO, mat.*,
                     ROUND(8 * mat.BASE * s.PESO_NEGOCIO * (0.6 + MOD(ABS(HASH(m.FECHA_PERIODO, s.ID_SOCIEDAD, mat.MATERIAL)), 80) / 100)
                           * IFF(m.meses_atras = 0 AND mat.TIPO_MATERIAL = 'Materia prima' AND s.ID_SOCIEDAD = 1000, 1.45, 1)) AS STOCK_LIB_UTIL,
                     ROUND(8 * mat.BASE * s.PESO_NEGOCIO * MOD(ABS(HASH(m.FECHA_PERIODO, s.ID_SOCIEDAD, mat.MATERIAL, 't')), 18) / 100) AS STOCK_TRANSITO,
                     ROUND(8 * mat.BASE * s.PESO_NEGOCIO * MOD(ABS(HASH(m.FECHA_PERIODO, s.ID_SOCIEDAD, mat.MATERIAL, 'b')), 4) / 100) AS STOCK_BLOQ
              FROM meses m CROSS JOIN {fq}.DIM_SOCIEDAD s CROSS JOIN mat)
            SELECT LEAST(LAST_DAY(FECHA_PERIODO), CURRENT_DATE()) AS FECHA_PERIODO, ID_SOCIEDAD AS SOCIEDAD, SOCIEDAD AS NOMBRE_SOCIEDAD, ID_PAIS AS PAIS,
                   ID_PAIS || '01' AS CENTRO, IFF(TIPO_MATERIAL = 'Producto terminado', 'PT01', 'MP01') AS ALMACEN,
                   MATERIAL, DESCRIPCION, TIPO_MATERIAL, UBM,
                   STOCK_LIB_UTIL, ROUND(STOCK_LIB_UTIL * COSTO_USD, 2) AS VALOR_LIB_UTIL,
                   STOCK_TRANSITO, ROUND(STOCK_TRANSITO * COSTO_USD, 2) AS VALOR_TRANSITO,
                   STOCK_BLOQ, ROUND(STOCK_BLOQ * COSTO_USD, 2) AS VALOR_STOCK_BLOQ,
                   ROUND((STOCK_LIB_UTIL + STOCK_TRANSITO + STOCK_BLOQ) * COSTO_USD, 2) AS VALOR_INVENTARIO_TOTAL,
                   (meses_atras = 0) AS ES_ULTIMO_CORTE
            FROM x""",
        # ---------- Cash flow (treasury movements) ----------
        f"""CREATE OR REPLACE TABLE {fq}.FLUJO_EFECTIVO COMMENT = 'Movimientos de tesorería: entradas y salidas de efectivo por concepto (USD y moneda local). Datos sintéticos.' AS
            WITH v AS (
              SELECT PERIODO, ID_SOCIEDAD, SUM(USD_VALOR_NETO) AS neto, SUM(USD_COSTO_VENTA) AS costo
              FROM {fq}.FACT_DETALLE_VENTAS GROUP BY 1, 2),
            conceptos AS (
              SELECT * FROM VALUES
                ('Entrada', 'Cobranza a clientes', 'neto', 0.97, 8),
                ('Entrada', 'Otros ingresos', 'neto', 0.012, 2),
                ('Entrada', 'Desembolso de préstamos', 'neto', 0.06, 1),
                ('Salida', 'Pago a proveedores', 'costo', 0.93, 6),
                ('Salida', 'Nómina y prestaciones', 'neto', 0.085, 2),
                ('Salida', 'Impuestos', 'neto', 0.07, 1),
                ('Salida', 'Fletes y distribución', 'neto', 0.06, 4),
                ('Salida', 'Servicios y mantenimiento', 'neto', 0.022, 3),
                ('Salida', 'Inversiones (CAPEX)', 'neto', 0.045, 2),
                ('Salida', 'Pago de deuda e intereses', 'neto', 0.035, 1)
                AS c(TIPO_MOVIMIENTO, CONCEPTO, base, pct, n_mov)),
            k AS (SELECT SEQ4() + 1 AS k FROM TABLE(GENERATOR(ROWCOUNT => 8))),
            x AS (
              SELECT v.PERIODO, v.ID_SOCIEDAD, c.TIPO_MOVIMIENTO, c.CONCEPTO, k.k, c.n_mov,
                     IFF(c.base = 'neto', v.neto, v.costo) * c.pct
                       * (0.85 + MOD(ABS(HASH(v.PERIODO, v.ID_SOCIEDAD, c.CONCEPTO)), 30) / 100)
                       * IFF(c.CONCEPTO = 'Inversiones (CAPEX)' AND v.PERIODO = DATE_TRUNC('month', CURRENT_DATE()) AND v.ID_SOCIEDAD = 1000, 3.2, 1)
                       / c.n_mov AS monto
              FROM v CROSS JOIN conceptos c JOIN k ON k.k <= c.n_mov
              WHERE NOT (c.CONCEPTO = 'Desembolso de préstamos' AND MOD(ABS(HASH(v.PERIODO, v.ID_SOCIEDAD)), 5) <> 0))
            SELECT 'M' || LPAD(ROW_NUMBER() OVER (ORDER BY x.PERIODO, x.ID_SOCIEDAD, CONCEPTO, k), 7, '0') AS ID_MOVIMIENTO,
                   DATEADD(day, MOD(ABS(HASH(x.PERIODO, x.ID_SOCIEDAD, CONCEPTO, k)), IFF(x.PERIODO = DATE_TRUNC('month', CURRENT_DATE()), DAY(CURRENT_DATE()), 28)), x.PERIODO) AS FECHA,
                   x.PERIODO, x.ID_SOCIEDAD, TIPO_MOVIMIENTO, CONCEPTO,
                   ARRAY_CONSTRUCT('Banco Popular', 'Banco Industrial', 'Banco Nacional', 'Banco del Caribe')[MOD(ABS(HASH(CONCEPTO, k)), 4)]::STRING AS BANCO,
                   ROUND(monto * (0.9 + MOD(ABS(HASH(x.PERIODO, CONCEPTO, k, 'm')), 20) / 100), 2) AS USD_MONTO,
                   s.MONEDA,
                   ROUND(monto * (0.9 + MOD(ABS(HASH(x.PERIODO, CONCEPTO, k, 'm')), 20) / 100) * s.TIPO_CAMBIO_USD, 2) AS MONTO_ML
            FROM x JOIN {fq}.DIM_SOCIEDAD s ON s.ID_SOCIEDAD = x.ID_SOCIEDAD""",
    ]

for stmt in load_statements(fq):
    session.sql(stmt).collect()

for t in ["DIM_SOCIEDAD", "DIM_PRODUCTO", "DIM_CLIENTE", "FACT_DETALLE_VENTAS",
          "ESTADO_RESULTADOS_MENSUAL", "INVENTARIOS_MB52", "FLUJO_EFECTIVO"]:
    n = session.sql(f"SELECT COUNT(*) FROM {fq}.{t}").collect()[0][0]
    print(f"{t:<28} {n:>9,} filas")

**1.3 · El modelo semántico.** Aquí está la clave del lab. Una *semantic view* le enseña a la IA **cómo habla el negocio**:
- "ventas" significa **venta neta** (bruta − descuentos − bonificaciones − rechazos);
- **margen bruto** = venta neta − costo de ventas;
- **EBITDA** = ventas netas − costo de ventas − gastos operativos (sin depreciación, intereses ni impuestos);
- el "valor del inventario" es la **foto más reciente**.

Sin estas definiciones, cada persona (y cada IA) calcularía las cifras a su manera. Con ellas, todos obtienen el mismo número.

In [ ]:
%%sql -r create_semantic_view
CREATE OR REPLACE SEMANTIC VIEW {{fq}}.SV_FINANZAS_ISM
  TABLES (
    sociedades AS {{fq}}.DIM_SOCIEDAD
      PRIMARY KEY (ID_SOCIEDAD)
      WITH SYNONYMS = ('empresa', 'compañía', 'país', 'company code', 'bukrs')
      COMMENT = 'Sociedades (company codes) de ISM: una por país',
    productos AS {{fq}}.DIM_PRODUCTO
      PRIMARY KEY (ID_SKU_PRODUCTO)
      WITH SYNONYMS = ('sku', 'artículo', 'producto terminado')
      COMMENT = 'Maestro de productos terminados con marca, categoría y formato',
    clientes AS {{fq}}.DIM_CLIENTE
      PRIMARY KEY (ID_CUENTA)
      WITH SYNONYMS = ('cliente', 'punto de venta', 'pdv', 'cuenta')
      COMMENT = 'Maestro de clientes con su canal y ciudad',
    ventas AS {{fq}}.FACT_DETALLE_VENTAS
      PRIMARY KEY (ID_VENTA)
      COMMENT = 'Ventas mensuales por cliente y SKU: bruto, descuentos, bonificaciones, rechazos, neto y costo de ventas (USD)',
    resultados AS {{fq}}.ESTADO_RESULTADOS_MENSUAL
      PRIMARY KEY (PERIODO, ID_SOCIEDAD, ORDEN)
      WITH SYNONYMS = ('estado de resultados', 'p&l', 'pyg', 'pérdidas y ganancias')
      COMMENT = 'Estado de resultados mensual por sociedad. Montos con signo: ingresos positivos, costos y gastos negativos (USD)',
    inventario AS {{fq}}.INVENTARIOS_MB52
      PRIMARY KEY (FECHA_PERIODO, SOCIEDAD, MATERIAL)
      WITH SYNONYMS = ('stock', 'existencias', 'mb52')
      COMMENT = 'Foto de inventario a fin de mes por sociedad y material (USD)',
    flujo AS {{fq}}.FLUJO_EFECTIVO
      PRIMARY KEY (ID_MOVIMIENTO)
      WITH SYNONYMS = ('caja', 'tesorería', 'efectivo', 'cash flow')
      COMMENT = 'Movimientos de tesorería: entradas y salidas de efectivo por concepto (USD)'
  )
  RELATIONSHIPS (
    ventas_sociedad AS ventas (ID_SOCIEDAD) REFERENCES sociedades,
    ventas_producto AS ventas (ID_SKU_PRODUCTO) REFERENCES productos,
    ventas_cliente AS ventas (ID_CUENTA) REFERENCES clientes,
    resultados_sociedad AS resultados (ID_SOCIEDAD) REFERENCES sociedades,
    inventario_sociedad AS inventario (SOCIEDAD) REFERENCES sociedades,
    flujo_sociedad AS flujo (ID_SOCIEDAD) REFERENCES sociedades
  )
  FACTS (
    ventas.venta_bruta_usd AS USD_VALOR_BRUTO COMMENT = 'Venta bruta a precio de lista (USD)',
    ventas.descuentos_usd AS USD_DESCUENTOS COMMENT = 'Descuentos comerciales (USD)',
    ventas.bonificaciones_usd AS USD_BONIFICACIONES COMMENT = 'Bonificaciones: producto entregado sin cargo o rebates (USD)',
    ventas.rechazos_usd AS USD_VALOR_RECHAZO COMMENT = 'Valor de la mercancía rechazada o devuelta (USD)',
    ventas.venta_neta_usd AS USD_VALOR_NETO COMMENT = 'Venta neta = bruta - descuentos - bonificaciones - rechazos (USD)',
    ventas.costo_venta_usd AS USD_COSTO_VENTA COMMENT = 'Costo de ventas de lo facturado (USD)',
    ventas.cajas AS CANTIDAD_CAJAS COMMENT = 'Cajas físicas vendidas',
    resultados.monto_usd AS USD_MONTO COMMENT = 'Monto con signo de la línea del estado de resultados (USD)',
    inventario.valor_inventario_usd AS VALOR_INVENTARIO_TOTAL COMMENT = 'Valor total del inventario: libre utilización + tránsito + bloqueado (USD)',
    flujo.monto_movimiento_usd AS USD_MONTO COMMENT = 'Monto del movimiento de efectivo, siempre positivo (USD)'
  )
  DIMENSIONS (
    sociedades.sociedad AS SOCIEDAD WITH SYNONYMS = ('empresa', 'compañía') COMMENT = 'Nombre de la sociedad: ISM Dominicana, ISM Guatemala, ISM Haití, ISM Perú, ISM Chile',
    sociedades.pais AS PAIS WITH SYNONYMS = ('country') COMMENT = 'País de la sociedad',
    sociedades.codigo_sociedad AS BUKRS_SOCIEDAD COMMENT = 'Código de sociedad SAP (BUKRS)',
    productos.producto AS PRODUCTO WITH SYNONYMS = ('sku', 'artículo') COMMENT = 'Nombre del producto terminado',
    productos.marca AS MARCA WITH SYNONYMS = ('brand') COMMENT = 'Marca: Kola Isla, Fresquita, Agua Montaña, Rayo, Frutalia, Hidra+, Té Brisa',
    productos.categoria AS CATEGORIA WITH SYNONYMS = ('categoría de producto', 'familia') COMMENT = 'Categoría: Gaseosas, Agua, Energizantes, Jugos y Néctares, Isotónicos, Té',
    productos.formato AS FORMATO WITH SYNONYMS = ('presentación', 'tamaño') COMMENT = 'Formato o presentación, por ejemplo 600ml, 2L',
    clientes.cliente AS NOMBRE_CLIENTE WITH SYNONYMS = ('nombre del cliente', 'punto de venta') COMMENT = 'Nombre del cliente',
    clientes.canal AS CANAL_CORPORATIVO WITH SYNONYMS = ('canal de venta', 'channel') COMMENT = 'Canal corporativo: Tradicional (colmados y tiendas), Moderno (supermercados), Mayorista, On Premise',
    clientes.ciudad AS CIUDAD COMMENT = 'Ciudad del cliente',
    ventas.mes_venta AS PERIODO WITH SYNONYMS = ('mes', 'periodo', 'month') COMMENT = 'Mes de la venta (primer día del mes)',
    ventas.es_mes_actual AS PERIODO = DATE_TRUNC('month', CURRENT_DATE()) WITH SYNONYMS = ('este mes', 'mes actual', 'mes en curso') COMMENT = 'TRUE para el mes en curso',
    ventas.es_mes_anterior AS PERIODO = DATEADD(month, -1, DATE_TRUNC('month', CURRENT_DATE())) WITH SYNONYMS = ('mes pasado', 'mes anterior') COMMENT = 'TRUE para el mes inmediatamente anterior al actual',
    resultados.mes_resultado AS PERIODO WITH SYNONYMS = ('mes', 'periodo') COMMENT = 'Mes del estado de resultados (primer día del mes)',
    resultados.es_mes_actual_resultado AS PERIODO = DATE_TRUNC('month', CURRENT_DATE()) WITH SYNONYMS = ('este mes', 'mes actual') COMMENT = 'TRUE para el mes en curso',
    resultados.linea_resultado AS LINEA_RESULTADO WITH SYNONYMS = ('línea', 'rubro', 'cuenta') COMMENT = 'Línea del estado de resultados: Ventas netas, Costo de ventas, Gastos de venta y mercadeo, Gastos de distribución, Gastos administrativos, Depreciación y amortización, Gastos financieros, Impuesto sobre la renta',
    resultados.orden_linea AS ORDEN COMMENT = 'Orden de presentación de la línea (1 = Ventas netas). Ordenar el estado de resultados por este campo',
    resultados.tipo_linea AS TIPO_LINEA COMMENT = 'Ingreso, Costo, Gasto operativo, Depreciación, Financiero o Impuesto',
    inventario.fecha_corte AS FECHA_PERIODO WITH SYNONYMS = ('fecha de inventario', 'corte') COMMENT = 'Fecha de la foto de inventario (fin de mes)',
    inventario.es_ultimo_corte AS ES_ULTIMO_CORTE WITH SYNONYMS = ('inventario actual', 'último inventario', 'hoy') COMMENT = 'TRUE para la foto de inventario más reciente. Usar siempre para "valor del inventario" actual',
    inventario.tipo_material AS TIPO_MATERIAL WITH SYNONYMS = ('tipo de material', 'mtart') COMMENT = 'Producto terminado, Materia prima, Envases y empaques, Repuestos, Material promocional',
    inventario.material AS DESCRIPCION WITH SYNONYMS = ('material', 'insumo') COMMENT = 'Descripción del material',
    inventario.centro AS CENTRO COMMENT = 'Centro (planta) SAP',
    flujo.fecha_movimiento AS FECHA COMMENT = 'Fecha del movimiento de efectivo',
    flujo.mes_flujo AS PERIODO WITH SYNONYMS = ('mes') COMMENT = 'Mes del movimiento de efectivo (primer día del mes)',
    flujo.es_mes_actual_flujo AS PERIODO = DATE_TRUNC('month', CURRENT_DATE()) WITH SYNONYMS = ('este mes', 'mes actual') COMMENT = 'TRUE para el mes en curso',
    flujo.tipo_movimiento AS TIPO_MOVIMIENTO WITH SYNONYMS = ('entrada o salida') COMMENT = 'Entrada o Salida de efectivo',
    flujo.concepto AS CONCEPTO WITH SYNONYMS = ('concepto de caja', 'motivo') COMMENT = 'Concepto: Cobranza a clientes, Otros ingresos, Desembolso de préstamos, Pago a proveedores, Nómina y prestaciones, Impuestos, Fletes y distribución, Servicios y mantenimiento, Inversiones (CAPEX), Pago de deuda e intereses',
    flujo.banco AS BANCO COMMENT = 'Banco del movimiento'
  )
  METRICS (
    ventas.venta_bruta AS SUM(USD_VALOR_BRUTO) WITH SYNONYMS = ('venta bruta', 'gross sales') COMMENT = 'Venta bruta (USD)',
    ventas.venta_neta AS SUM(USD_VALOR_NETO) WITH SYNONYMS = ('ventas', 'cuánto vendimos', 'venta neta', 'net sales', 'facturación') COMMENT = 'Venta neta (USD). Es la medida oficial de "ventas"',
    ventas.descuentos AS SUM(USD_DESCUENTOS) COMMENT = 'Descuentos (USD)',
    ventas.bonificaciones AS SUM(USD_BONIFICACIONES) WITH SYNONYMS = ('bonificación', 'rebates') COMMENT = 'Bonificaciones (USD)',
    ventas.descuentos_y_bonificaciones AS SUM(USD_DESCUENTOS + USD_BONIFICACIONES) COMMENT = 'Descuentos + bonificaciones (USD)',
    ventas.pct_descuentos_bonificaciones AS SUM(USD_DESCUENTOS + USD_BONIFICACIONES) / NULLIF(SUM(USD_VALOR_BRUTO), 0) WITH SYNONYMS = ('peso de descuentos', 'cuánto representan los descuentos') COMMENT = 'Descuentos + bonificaciones como fracción de la venta bruta (0.17 = 17%)',
    ventas.costo_ventas AS SUM(USD_COSTO_VENTA) COMMENT = 'Costo de ventas (USD)',
    ventas.margen_bruto AS SUM(USD_VALOR_NETO - USD_COSTO_VENTA) WITH SYNONYMS = ('margen', 'utilidad bruta', 'margen de contribución') COMMENT = 'Margen bruto = venta neta - costo de ventas (USD)',
    ventas.pct_margen_bruto AS SUM(USD_VALOR_NETO - USD_COSTO_VENTA) / NULLIF(SUM(USD_VALOR_NETO), 0) WITH SYNONYMS = ('margen %', 'margen porcentual') COMMENT = 'Margen bruto como fracción de la venta neta (0.35 = 35%)',
    ventas.cajas_vendidas AS SUM(CANTIDAD_CAJAS) WITH SYNONYMS = ('volumen', 'cajas') COMMENT = 'Cajas físicas vendidas',
    ventas.venta_neta_mes_actual AS SUM(IFF(PERIODO = DATE_TRUNC('month', CURRENT_DATE()), USD_VALOR_NETO, 0)) COMMENT = 'Venta neta del mes en curso (USD)',
    ventas.venta_neta_mes_anterior AS SUM(IFF(PERIODO = DATEADD(month, -1, DATE_TRUNC('month', CURRENT_DATE())), USD_VALOR_NETO, 0)) COMMENT = 'Venta neta del mes anterior (USD)',
    ventas.variacion_venta_vs_mes_anterior AS SUM(IFF(PERIODO = DATE_TRUNC('month', CURRENT_DATE()), USD_VALOR_NETO, 0)) - SUM(IFF(PERIODO = DATEADD(month, -1, DATE_TRUNC('month', CURRENT_DATE())), USD_VALOR_NETO, 0))
      WITH SYNONYMS = ('caída en ventas', 'variación de ventas') COMMENT = 'Venta neta del mes actual menos la del mes anterior (USD). Negativo = caída. Ordenar ascendente para ver las mayores caídas',
    ventas.pct_variacion_venta_vs_mes_anterior AS (SUM(IFF(PERIODO = DATE_TRUNC('month', CURRENT_DATE()), USD_VALOR_NETO, 0)) - SUM(IFF(PERIODO = DATEADD(month, -1, DATE_TRUNC('month', CURRENT_DATE())), USD_VALOR_NETO, 0)))
        / NULLIF(SUM(IFF(PERIODO = DATEADD(month, -1, DATE_TRUNC('month', CURRENT_DATE())), USD_VALOR_NETO, 0)), 0)
      COMMENT = 'Variación porcentual de la venta neta del mes actual vs el mes anterior (-0.30 = cayó 30%)',
    resultados.ingresos AS SUM(IFF(TIPO_LINEA = 'Ingreso', USD_MONTO, 0)) WITH SYNONYMS = ('ingresos', 'revenue') COMMENT = 'Ingresos (ventas netas) del estado de resultados (USD)',
    resultados.costos AS -SUM(IFF(TIPO_LINEA = 'Costo', USD_MONTO, 0)) COMMENT = 'Costo de ventas, en positivo (USD)',
    resultados.gastos AS -SUM(IFF(TIPO_LINEA NOT IN ('Ingreso', 'Costo'), USD_MONTO, 0)) COMMENT = 'Todos los gastos (operativos, depreciación, financieros e impuestos), en positivo (USD)',
    resultados.gastos_operativos AS -SUM(IFF(TIPO_LINEA = 'Gasto operativo', USD_MONTO, 0)) COMMENT = 'Gastos de venta, distribución y administración, en positivo (USD)',
    resultados.ebitda AS SUM(IFF(INCLUIDO_EN_EBITDA, USD_MONTO, 0)) WITH SYNONYMS = ('ebitda', 'resultado operativo antes de depreciación') COMMENT = 'EBITDA = ventas netas - costo de ventas - gastos operativos (USD)',
    resultados.pct_margen_ebitda AS SUM(IFF(INCLUIDO_EN_EBITDA, USD_MONTO, 0)) / NULLIF(SUM(IFF(TIPO_LINEA = 'Ingreso', USD_MONTO, 0)), 0) COMMENT = 'EBITDA como fracción de las ventas netas',
    resultados.utilidad_neta AS SUM(USD_MONTO) WITH SYNONYMS = ('resultado del mes', 'utilidad', 'ganancia', 'net income', 'resultado neto') COMMENT = 'Utilidad neta = suma de todas las líneas del estado de resultados (USD)',
    resultados.monto_linea AS SUM(USD_MONTO) COMMENT = 'Monto de una línea del estado de resultados. Usar con linea_resultado y orden_linea para mostrar el estado de resultados',
    inventario.valor_inventario AS SUM(VALOR_INVENTARIO_TOTAL) WITH SYNONYMS = ('valor del inventario', 'valor del stock') COMMENT = 'Valor del inventario (USD). Filtrar es_ultimo_corte = TRUE para el valor actual',
    flujo.entradas_efectivo AS SUM(IFF(TIPO_MOVIMIENTO = 'Entrada', USD_MONTO, 0)) WITH SYNONYMS = ('entradas', 'ingresos de caja', 'cobros') COMMENT = 'Entradas de efectivo (USD)',
    flujo.salidas_efectivo AS SUM(IFF(TIPO_MOVIMIENTO = 'Salida', USD_MONTO, 0)) WITH SYNONYMS = ('salidas', 'egresos', 'pagos') COMMENT = 'Salidas de efectivo (USD)',
    flujo.flujo_neto AS SUM(IFF(TIPO_MOVIMIENTO = 'Entrada', USD_MONTO, -USD_MONTO)) WITH SYNONYMS = ('flujo neto', 'flujo de caja neto') COMMENT = 'Entradas menos salidas de efectivo (USD)'
  )
  COMMENT = 'Modelo financiero de ISM (datos sintéticos): ventas, margen, estado de resultados, inventario y flujo de efectivo. Montos en USD'

**1.4 · El agente.** Crea el **Asistente Financiero ISM**: entiende la pregunta, consulta el modelo semántico y responde en español, en tono ejecutivo.

In [ ]:
%%sql -r create_agent
CREATE OR REPLACE AGENT {{fq}}.ASISTENTE_FINANCIERO_ISM
  COMMENT = 'ISM Snow Day: responde preguntas financieras y comerciales de gerencia'
  PROFILE = '{"display_name": "Asistente Financiero ISM"}'
  FROM SPECIFICATION
  $$
  models:
    orchestration: auto

  instructions:
    response: "Eres el asistente financiero de la gerencia de ISM. Responde siempre en español, de forma ejecutiva: primero la cifra clave en una frase, luego una tabla corta. Todos los montos están en USD; muéstralos con separador de miles y sin decimales, y los porcentajes con un decimal. Si la pregunta dice 'este mes' o 'el mes', usa el mes en curso. Los datos del mes en curso son el cierre preliminar del mes: no adviertas que el mes está incompleto ni que las fechas son futuras. Cierra con una observación de negocio breve."
    orchestration: "Usa la herramienta finanzas_ism para toda pregunta sobre ventas, margen, descuentos, bonificaciones, estado de resultados, EBITDA, inventario o flujo de efectivo. 'Ventas' significa venta neta. El valor del inventario actual se filtra con es_ultimo_corte. Para el estado de resultados muestra las líneas ordenadas por orden_linea e incluye la utilidad neta al final."
    sample_questions:
      - question: "¿Cuál es el margen por categoría este mes?"
      - question: "¿Cuál es el resultado del mes y el EBITDA por sociedad?"
      - question: "¿Cuánto representan los descuentos y bonificaciones en el mes actual?"
      - question: "¿Qué productos tuvieron la mayor caída en ventas?"
      - question: "¿Cuáles fueron las entradas y salidas de efectivo del mes, por concepto?"

  tools:
    - tool_spec:
        type: "cortex_analyst_text_to_sql"
        name: "finanzas_ism"
        description: "Responde preguntas sobre ventas, margen, descuentos y bonificaciones por sociedad, producto, categoría, canal y cliente; estado de resultados y EBITDA por sociedad; valor del inventario por sociedad y tipo de material; y entradas y salidas de efectivo por concepto."

  tool_resources:
    finanzas_ism:
      semantic_view: "{{fq}}.SV_FINANZAS_ISM"
      execution_environment:
        type: "warehouse"
        warehouse: "{{wh}}"
  $$

**1.5 · Publicarlo en CoWork.**

In [ ]:
agent_name = "ASISTENTE_FINANCIERO_ISM"
display_name = "Asistente Financiero ISM"

agent_fqn = f"{fq}.{agent_name}"
si_objects = session.sql("SHOW SNOWFLAKE INTELLIGENCES").collect()

if not si_objects:
    print("Esta cuenta no tiene objeto CoWork: los agentes aparecen en CoWork automáticamente.")
else:
    si_name = si_objects[0]["name"]
    try:
        session.sql(f"ALTER SNOWFLAKE INTELLIGENCE {si_name} ADD AGENT {agent_fqn}").collect()
        print(f"Agente {agent_fqn} agregado a CoWork ({si_name}).")
    except Exception as e:
        if "already" in str(e).lower():
            print(f"El agente {agent_fqn} ya estaba en CoWork ({si_name}).")
        else:
            print(f"No se pudo agregar el agente a {si_name}. Verifique que la celda de setup corrió como ACCOUNTADMIN.\n{e}")

print(f"Abra CoWork en https://ai.snowflake.com y busque '{display_name}'.")

## Paso 2 · Pregúntele al asistente (12 min)

1. Abra **CoWork**: https://ai.snowflake.com (o, en Snowsight, **AI & ML → CoWork**).
2. Elija el agente **Asistente Financiero ISM**.
3. Copie y pegue estas preguntas, una por una:

| # | Pregunta para el asistente |
|---|---|
| 1 | ¿Cuál es el margen por categoría este mes? ¿Y por canal? |
| 2 | ¿Cuál es el resultado del mes y el EBITDA por sociedad? |
| 3 | ¿Cuánto representan los descuentos y bonificaciones en el mes actual? |
| 4 | ¿Cuánto vendimos este mes por sociedad, categoría de producto y cliente? Muéstrame los 10 clientes principales. |
| 5 | ¿Cuáles fueron las entradas y salidas de efectivo del mes? |
| 6 | ¿Qué productos o clientes tuvieron la mayor caída en ventas frente al mes anterior? |
| 7 | ¿Cuál es el resultado del mes (ingresos, costos, gastos y utilidad)? |
| 8 | ¿Cuál es el valor del inventario por sociedad y tipo de material? |
| 9 | ¿Cuáles fueron las entradas y salidas de efectivo del mes, por concepto? |

**Pruebe también:** "Hazme un gráfico de la venta neta mensual por sociedad del último año" o "¿Por qué cayó Rayo Energy 473ml?".

> 💡 **Dato:** en la respuesta, abra el detalle para ver el SQL que generó el asistente. Usó las definiciones del modelo semántico, no una adivinanza.

## Paso 3 · Compare con las respuestas esperadas (3 min)

Cada celda calcula la respuesta directamente desde el modelo semántico. Ejecútelas y compárelas con lo que respondió el asistente. **Deben coincidir.**

In [ ]:
%%sql -r esperada_1_margen_categoria
-- Pregunta 1 · Margen por categoría, mes actual
SELECT * FROM SEMANTIC_VIEW(
    {{fq}}.SV_FINANZAS_ISM
    DIMENSIONS productos.categoria
    METRICS ventas.venta_neta, ventas.margen_bruto, ventas.pct_margen_bruto
    WHERE ventas.es_mes_actual
)
ORDER BY margen_bruto DESC

In [ ]:
%%sql -r esperada_2_ebitda_sociedad
-- Pregunta 2 · Resultado del mes y EBITDA por sociedad
SELECT * FROM SEMANTIC_VIEW(
    {{fq}}.SV_FINANZAS_ISM
    DIMENSIONS sociedades.sociedad
    METRICS resultados.ingresos, resultados.ebitda, resultados.pct_margen_ebitda, resultados.utilidad_neta
    WHERE resultados.es_mes_actual_resultado
)
ORDER BY ebitda DESC

In [ ]:
%%sql -r esperada_3_descuentos
-- Pregunta 3 · Peso de descuentos y bonificaciones, mes actual
SELECT * FROM SEMANTIC_VIEW(
    {{fq}}.SV_FINANZAS_ISM
    METRICS ventas.venta_bruta, ventas.descuentos, ventas.bonificaciones, ventas.pct_descuentos_bonificaciones
    WHERE ventas.es_mes_actual
)

In [ ]:
%%sql -r esperada_6_caidas
-- Pregunta 6 · Productos con mayor caída vs el mes anterior
SELECT * FROM SEMANTIC_VIEW(
    {{fq}}.SV_FINANZAS_ISM
    DIMENSIONS productos.producto
    METRICS ventas.venta_neta_mes_anterior, ventas.venta_neta_mes_actual, ventas.pct_variacion_venta_vs_mes_anterior
)
ORDER BY pct_variacion_venta_vs_mes_anterior
LIMIT 5

In [ ]:
%%sql -r esperada_7_estado_resultados
-- Pregunta 7 · Estado de resultados del mes (todas las sociedades)
SELECT * FROM SEMANTIC_VIEW(
    {{fq}}.SV_FINANZAS_ISM
    DIMENSIONS resultados.orden_linea, resultados.linea_resultado
    METRICS resultados.monto_linea
    WHERE resultados.es_mes_actual_resultado
)
ORDER BY orden_linea

In [ ]:
%%sql -r esperada_8_inventario
-- Pregunta 8 · Valor del inventario por sociedad y tipo de material
SELECT * FROM SEMANTIC_VIEW(
    {{fq}}.SV_FINANZAS_ISM
    DIMENSIONS sociedades.sociedad, inventario.tipo_material
    METRICS inventario.valor_inventario
    WHERE inventario.es_ultimo_corte
)
ORDER BY sociedad, valor_inventario DESC

In [ ]:
%%sql -r esperada_9_flujo
-- Preguntas 5 y 9 · Entradas y salidas de efectivo del mes, por concepto
SELECT * FROM SEMANTIC_VIEW(
    {{fq}}.SV_FINANZAS_ISM
    DIMENSIONS flujo.tipo_movimiento, flujo.concepto
    METRICS flujo.entradas_efectivo, flujo.salidas_efectivo
    WHERE flujo.es_mes_actual_flujo
)
ORDER BY tipo_movimiento, entradas_efectivo + salidas_efectivo DESC

### Cierre: un resumen ejecutivo

Pídale al asistente en CoWork:

> *"Prepárame un resumen ejecutivo del cierre del mes para el comité de gerencia: ventas, margen, EBITDA por sociedad, descuentos y bonificaciones, inventario y caja. Incluye 3 alertas y 3 recomendaciones."*

## Lo que se lleva

- **La IA responde bien cuando el negocio define las reglas.** El modelo semántico es donde viven "venta neta", "margen" y "EBITDA". Si mañana cambia una definición, se cambia una vez y todos los reportes y agentes se actualizan.
- **Las preguntas de gerencia ya no esperan un reporte.** Las mismas 9 preguntas del cierre se responden en segundos, por sociedad, canal o cliente.
- **Con sus datos reales**, este mismo patrón se conecta a las tablas que ISM ya tiene en Snowflake (ventas SAP, cuenta mayor, MB52, tesorería).

¿Le quedó tiempo? Pregúntele al asistente algo que su equipo pida cada mes y vea si lo responde. Si no puede, es una pista de qué agregar al modelo semántico.

---
### Limpieza (opcional)
Su cuenta de lab se elimina al final del evento. Si quiere borrar los objetos antes, quite el `--` y ejecute la celda.

In [ ]:
%%sql -r limpieza
-- DROP SCHEMA IF EXISTS {{fq}};